# Hand-Built CNN Feature Extractor
Implement `convolution -> ReLU -> max-pooling` in NumPy and classify the scikit-learn digits dataset.


In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [2]:
def load_images():
    d = load_digits()
    X_img = d.images.astype(float)
    y = d.target
    return X_img, y


def conv2d_valid(img, kernel):
    img = np.asarray(img, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    H, W = img.shape
    kh, kw = kernel.shape
    out = np.zeros((H - kh + 1, W - kw + 1), dtype=float)

    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            region = img[i:i + kh, j:j + kw]
            out[i, j] = np.sum(region * kernel)

    return out


def relu(x):
    x = np.asarray(x, dtype=float)
    return np.maximum(0, x)


def max_pool2d(fmap, size=2):
    fmap = np.asarray(fmap, dtype=float)
    H, W = fmap.shape
    out = np.zeros((H // size, W // size), dtype=float)

    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            block = fmap[i*size:(i+1)*size, j*size:(j+1)*size]
            out[i, j] = np.max(block)

    return out


def extract_features(X_img, kernels, pool=2):
    all_features = []

    for img in X_img:
        image_features = []
        for kernel in kernels:
            conv_map = conv2d_valid(img, kernel)
            activated_map = relu(conv_map)
            pooled_map = max_pool2d(activated_map, pool)
            image_features.append(pooled_map.flatten())

        all_features.append(np.concatenate(image_features))

    return np.asarray(all_features, dtype=float)


def default_kernels():
    sobel_x = np.array([[1, 0, -1],
                        [2, 0, -2],
                        [1, 0, -1]], dtype=float)
    sobel_y = np.array([[1, 2, 1],
                        [0, 0, 0],
                        [-1, -2, -1]], dtype=float)
    laplacian = np.array([[0, -1, 0],
                          [-1, 4, -1],
                          [0, -1, 0]], dtype=float)
    blur = np.ones((3, 3), dtype=float) / 9
    return [sobel_x, sobel_y, laplacian, blur]


In [3]:
img = np.arange(9.).reshape(3, 3)
print(conv2d_valid(img, np.array([[1, 0], [0, -1.]])))
print(relu(np.array([-2., 0., 3.5])))
print(max_pool2d(np.arange(1, 17.).reshape(4, 4), 2))
X_img, y = load_images()
F = extract_features(X_img[:5], default_kernels(), pool=2)
print(X_img.shape, y.shape)
print(F.shape)


[[-4. -4.]
 [-4. -4.]]
[0.  0.  3.5]
[[ 6.  8.]
 [14. 16.]]
(1797, 8, 8) (1797,)
(5, 36)


In [4]:
X_img, y = load_images()
features = extract_features(X_img, default_kernels(), pool=2)
X_train, X_test, y_train, y_test = train_test_split(
    features, y, test_size=0.2, random_state=42, stratify=y
)
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print("Feature matrix shape:", features.shape)
print("Test accuracy:", accuracy)


Feature matrix shape: (1797, 36)
Test accuracy: 0.9555555555555556


In [5]:
# Autograder-aligned checks
X_check, y_check = load_images()
assert X_check.shape == (1797, 8, 8) and y_check.shape == (1797,)
assert np.issubdtype(X_check.dtype, np.floating)
assert np.allclose(conv2d_valid(np.arange(9.).reshape(3, 3), np.array([[1, 0], [0, -1.]])), -4)
assert conv2d_valid(np.zeros((8, 8)), np.ones((3, 3))).shape == (6, 6)
assert np.array_equal(relu([-2., 0., 3.5]), np.array([0., 0., 3.5]))
assert np.array_equal(max_pool2d(np.arange(1, 17.).reshape(4, 4), 2), np.array([[6., 8.], [14., 16.]]))
assert extract_features(X_check[:20], default_kernels()[:3], pool=2).shape == (20, 27)
assert accuracy > 0.90
print("All checks passed.")


All checks passed.
